# 複数のプロンプトをまとめてバッチ生成する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_llm/llm_batch_generation.ipynb)

NLP クックブック 12-03 ／ 大規模言語モデル (LLM)

大量の入力を LLM で処理したい。1 件ずつ `generate` すると遅いので、バッチでまとめて生成したい。

In [1]:
import sentencepiece as spm
import time
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
model = AutoModelForCausalLM.from_pretrained(name).eval()
tokenizer.padding_side = "left"               # 生成モデルのバッチ処理では必ず左パディング
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

words = ["形態素解析", "機械翻訳", "固有表現抽出", "要約", "質問応答", "感情分析", "音声認識", "情報検索"]
prompts = [tokenizer.apply_chat_template([{"role": "user", "content": f"「{w}」を一文で説明してください。"}],
                                         add_generation_prompt=True, tokenize=False) for w in words]

@torch.no_grad()
def generate(batch):
    enc = tokenizer(batch, return_tensors="pt", padding=True, add_special_tokens=False)
    out = model.generate(**enc, max_new_tokens=40, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

t = time.perf_counter(); single = [generate([p])[0] for p in prompts]; t1 = time.perf_counter() - t
t = time.perf_counter(); batched = generate(prompts); t2 = time.perf_counter() - t
print(f"1 件ずつ: {t1:.1f} 秒 / バッチ: {t2:.1f} 秒")
for w, s in zip(words, batched):
    print(f"{w}: {s.strip()}")

1 件ずつ: 15.4 秒 / バッチ: 8.1 秒
形態素解析: 「形態素解析」は、テキストデータをその単語やフレーズの最小単位である「形態素」に分割する処理を指します。これにより、文章の意味を解析し、単語や品詞ごとに
機械翻訳: 「機械翻訳」とは、コンピューターを用いて自然言語を別の言語に翻訳する技術を指します。この技術は、大量のデータと機械学習アルゴリズムを利用して、翻訳の精度を向上させています。機械翻訳は、特にビジネス
固有表現抽出: 「固有表現抽出」とは、テキストデータから特定の意味を持つ単語やフレーズ（固有表現）を抽出し、その情報を活用するプロセスを指します。これにより、テキスト内の重要な概念やキーワードを特定し、分析
要約: 「要約」とは、ある文章や情報から要点を抽出し、簡潔にまとめることを指します。これにより、元の内容を簡単に理解したり、重要な部分を整理したりすることができます。
質問応答: 「質問応答」は、ユーザー asked に対する回答やフィードバックを提供するプロセスを指します。これは、自然言語処理や人工知能技術を用いて、ユーザーが入力した質問に対して適切な情報や回答を提供することです。これにより、ユーザー
感情分析: 「感情分析」は、テキストデータから感情を特定し、その感情の強度や種類を分析するプロセスを指します。この手法は、文章や会話における感情的なニュアンスを理解するために広く用いられています。
音声認識: 音声認識とは、音声データをコンピュータが認識し、テキストやコマンドに変換する技術です。これにより、音声入力や音声コマンドによる操作が可能になり、様々なアプリケーションやデバイスで活用されています。
情報検索: 「情報検索」とは、特定のテーマや目的に基づいて大量の情報の中から必要な情報を効率的に探し出すプロセスを指します。これはコンピュータやインターネットを利用して行われることが多く、様々な情報源から得たデータを整理・分析し、有用な


- 右パディングのままだと、短いプロンプトの後ろにパディングが入り、その続きを生成することになって出力が壊れる。
- バッチ内の長さを揃える（長さでソートする）とパディングが減って効率が良い。
- 数万件以上を本格的に処理するなら、vLLM などの推論サーバー（連続バッチ処理）が桁違いに速い。